## Download Dataset

In [ ]:
import gdown

# 2. Extract your File ID from your public Drive link:
# Format: https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing
file_id = "1hEk9EmMmLzURNkpYFDi0uDC7YDYjbDG0"
url = f"https://drive.google.com/uc?id={file_id}"
output_path = "./data.zip"

# 3. Download the file
gdown.download(url, output_path, quiet=False)

In [ ]:
# -q: quiet (suppresses printing thousands of file paths)
# -o: overwrite existing files without prompting
!unzip -q -o ./data.zip -d ./

# Remove the original zip to free up disk space
!rm ./data.zip

## Download Class Weights

In [ ]:
import gdown

# 2. Extract your File ID from your public Drive link:
# Format: https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing
file_id = "1OIFa4WG6fTU9VPXY2iigMNV_c0XQaKeJ"
url = f"https://drive.google.com/uc?id={file_id}"
output_path = "./data/class_weights.npy"

# 3. Download the file
gdown.download(url, output_path, quiet=False)

## Import

In [ ]:
!pip install torchvision pillow torchinfo matplotlib seaborn graphviz segmentation-models-pytorch jaxtyping beartype tqdm psutil


In [ ]:
import gc
import glob
import os
import psutil
import shutil
from typing import Any, Callable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torchvision
import torch.optim.lr_scheduler as lr_scheduler

from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchinfo import summary
from torchvision import tv_tensors
from torchvision.transforms import v2

from PIL import Image
from tqdm import tqdm
from typing import Dict, List, cast

from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp

from jaxtyping import Float, Int64, UInt8
from beartype import beartype


## Configurations

In [ ]:
class Configuration:
    """Global configuration settings for data loading, training, and validation."""

    # Set to True when debugging locally to validate tensor layouts and dtypes.
    # Set to False on Kaggle/remote training to prevent beartype/jaxtyping dynamic
    # wrapper allocations from leaking host CPU RAM across long training sessions.
    SHOULD_CHECK: bool = False

    DEVICE: torch.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    NUM_DEVICES: int = torch.cuda.device_count()
    # Start with no worker processes to remove host-RAM/shared-memory pressure from multiprocessing.
    NUM_WORKERS: int = 0
    MEMORY_LOG_INTERVAL: int = 100

    NUM_CLASSES: int = 20
    EPOCHS: int = 20
    # Stability baseline: use one T4 and batch size 8. nn.DataParallel is intentionally disabled.
    # Once host RSS is proven flat, multi-GPU should be implemented with DDP rather than DataParallel.
    BATCH_SIZE: int = 8
    ACCUMULATION_STEPS: int = 1
    LR: float = 1e-4
    PATIENCE: int = 8

    APPLY_SHUFFLE: bool = True
    SEED: int = 768

    # Spatial dimensions of raw BDD100k images before preprocessing
    ORIGINAL_IMAGE_HEIGHT: int = 720
    ORIGINAL_IMAGE_WIDTH: int = 1280

    # Downsampled resolution chosen to balance spatial detail against GPU memory consumption
    IMAGE_HEIGHT: int = 360
    IMAGE_WIDTH: int = 640
    CHANNELS: int = 3  # Standard RGB channels


class Path:
    """Filesystem path definitions for datasets, masks, and precomputed weights."""

    BASE: str = "./data/bdd100k"

    SEGMENTATION_MASK_LABEL_FOLDER: str = BASE + "/segmentation_maps/color_labels"
    SEGMENTATION_MASK_TRAIN_PATH: str = SEGMENTATION_MASK_LABEL_FOLDER + "/train"
    SEGMENTATION_MASK_VAL_PATH: str = SEGMENTATION_MASK_LABEL_FOLDER + "/val"

    IMAGE_FOLDER: str = BASE + "/images_10k"
    IMAGE_TRAIN_PATH: str = IMAGE_FOLDER + "/train"
    IMAGE_VAL_PATH: str = IMAGE_FOLDER + "/val"

    CLASS_WEIGHTS_PATH: str = "./data/class_weights.npy"

## Tensor Declarations

In [ ]:
# Shape aliases that document the tensor layout at each stage of the pipeline.
#
# ``h``/``w`` are the spatial dimensions, ``b`` is the batch size and ``c`` is
# the number of channels/classes.
ImageTensor = Float[Tensor, "3 h w"]  # single image, channel-first layout
IndexMaskTensor = Int64[Tensor, "h w"]
MaskTensor = Float[Tensor, "c h w"]  # one-hot mask, ``c == NUM_CLASSES``
BatchImage = Float[Tensor, "b 3 h w"]  # collated batch of images
BatchIndexMask = Int64[Tensor, "b h w"]
Logits = Float[Tensor, "b c h w"]  # model output, ``c == NUM_CLASSES``
ClassMask = Float[Tensor, "c h w"]  # per-sample probability/binary mask, ``c`` channels
Scalar = Float[Tensor, ""]  # scalar (0-dim) tensor
NumpyImage = Float[np.ndarray, "h w 3"]  # single image, channels-last layout
ClassIndexArray = UInt8[np.ndarray, "h w"]  # per-pixel class id map (numpy)
RawColorImage = UInt8[np.ndarray, "h w 3"]  # single image, channel-last layout

## Class Colors

In [ ]:
# BDD100k color-label palette. The row index is the class id (0-19), matching
# ``Configuration.NUM_CLASSES``. Predicted class maps are coloured with this
# same palette so they render side by side with the ground-truth color labels
# stored on disk. The exact colour per class only needs to be distinct and
# consistent; it mirrors the default BDD100k colours.
CLASS_COLORS: np.ndarray = np.array(
    [
        [128, 64, 128],  # 0  - Road
        [244, 35, 232],  # 1  - Sidewalk
        [70, 70, 70],  # 2  - Building
        [102, 102, 156],  # 3  - Wall
        [190, 153, 153],  # 4  - Fence
        [153, 153, 153],  # 5  - Pole
        [250, 170, 30],  # 6  - Traffic Light
        [220, 220, 0],  # 7  - Traffic Sign
        [107, 142, 35],  # 8  - Vegetation
        [152, 251, 152],  # 9  - Terrain
        [70, 130, 180],  # 10 - Sky
        [220, 20, 60],  # 11 - Person
        [255, 0, 0],  # 12 - Rider
        [0, 0, 142],  # 13 - Car
        [0, 0, 70],  # 14 - Truck
        [0, 60, 100],  # 15 - Bus
        [0, 80, 100],  # 16 - Train
        [0, 0, 230],  # 17 - Motorcycle
        [119, 11, 32],  # 18 - Bicycle
        [0, 0, 0],  # 19 - Unknown
    ],
    dtype=np.uint8,
)

CLASS_NAMES: list[str] = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train", "motorcycle",
    "bicycle", "unknown",
]


def color_label_to_class_index(label: RawColorImage) -> ClassIndexArray:
    """Map an RGB color-label image to a per-pixel class-index map.

    BDD100k stores segmentation masks as RGB PNGs[cite: 1]. This function converts
    the 3-channel colour values to integer class indices expected by the loss
    function and evaluation metrics. Vectorized 32-bit integer encoding maps
    RGB tuples into single scalar keys to eliminate intermediate array allocations.

    Steps
    -----
    1. Initialize the class index map with the index of the 'unknown' class.
    2. Pack RGB triplets into a single 32-bit integer using bit-shifts.
    3. Pack palette colors using the identical bit-shift formula.
    4. Match packed pixel integers to palette entries and assign class indices.

    Parameters
    ----------
    label : RawColorImage
        RGB color-label array of shape ``(H, W, 3)`` and dtype ``uint8``.

    Returns
    -------
    ClassIndexArray
        Class-index array of shape ``(H, W)`` and dtype ``uint8`` with values in ``[0, 19]``.
    """
    # Initialize all pixels to the fallback Unknown class (index 19)
    class_ids = np.full(label.shape[:2], len(CLASS_COLORS) - 1, dtype=np.uint8)

    # Pack RGB channels into a single int32 scalar: (R << 16) | (G << 8) | B
    # This enables single-operation comparisons without allocating 20 temporary boolean arrays
    label_packed = (
        (label[..., 0].astype(np.int32) << 16)
        | (label[..., 1].astype(np.int32) << 8)
        | label[..., 2].astype(np.int32)
    )
    palette_packed = (
        (CLASS_COLORS[:, 0].astype(np.int32) << 16)
        | (CLASS_COLORS[:, 1].astype(np.int32) << 8)
        | CLASS_COLORS[:, 2].astype(np.int32)
    )

    # Assign class indices where packed pixel keys match palette entries
    for class_id, color_key in enumerate(palette_packed):
        class_ids[label_packed == color_key] = class_id

    return class_ids

## Dataset

In [ ]:
class BDDSegmentationDataset(Dataset[tuple[ImageTensor, IndexMaskTensor]]):
    """BDD100K semantic-segmentation dataset with torchvision-v2 transforms.

    This implementation deliberately avoids Albumentations in the hot data-loading
    path. Images stay uint8 until after resizing, which avoids constructing a
    full-resolution float32 array for every sample. Segmentation masks are wrapped
    as ``tv_tensors.Mask`` so torchvision v2 applies geometric transforms such as
    resize and horizontal flip consistently to the image and mask while preserving
    nearest-neighbour semantics for the mask.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        transform: Callable[..., tuple[Tensor, Tensor]] | None = None,
    ) -> None:
        """Initialize file references and an optional paired transform.

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame containing ``image_paths`` and ``mask_paths`` columns.
        transform : callable or None, optional
            A torchvision-v2 transform accepting ``(image, mask)`` and returning
            transformed ``(image, mask)`` tensors.
        """
        super().__init__()
        self.image_paths: list[str] = df["image_paths"].to_list()
        self.mask_paths: list[str] = df["mask_paths"].to_list()
        self.transform = transform

    def load_sample(self, index: int) -> tuple[RawColorImage, ClassIndexArray]:
        """Load a raw RGB image and convert its color mask to class indices.

        The image is intentionally kept as uint8 here. Converting to float32 is
        postponed until after resizing in ``__getitem__`` so each sample uses
        substantially less transient host memory.

        Parameters
        ----------
        index : int
            Dataset index.

        Returns
        -------
        tuple[RawColorImage, ClassIndexArray]
            RGB uint8 image with shape ``(H, W, 3)`` and uint8 class-index mask
            with shape ``(H, W)``.
        """
        if index < 0 or index >= len(self.image_paths):
            raise IndexError(
                f"Index {index} is out of bounds for dataset of length {len(self)}."
            )

        image_path = self.image_paths[index]
        mask_path = self.mask_paths[index]

        with Image.open(image_path) as image_pil:
            image = np.array(image_pil.convert("RGB"), dtype=np.uint8)

        with Image.open(mask_path) as mask_pil:
            raw_mask = np.array(mask_pil.convert("RGB"), dtype=np.uint8)

        class_mask = color_label_to_class_index(raw_mask)

        # raw_mask is no longer needed after class-id conversion.
        del raw_mask

        return image, class_mask

    def __len__(self) -> int:
        """Return the number of paired image/mask samples."""
        return len(self.image_paths)

    def __getitem__(self, index: int) -> tuple[ImageTensor, IndexMaskTensor]:
        """Load one sample and apply memory-stable paired torchvision transforms.

        Returns
        -------
        tuple[ImageTensor, IndexMaskTensor]
            Float32 image tensor ``(3, H, W)`` in ``[0, 1]`` and int64 mask
            tensor ``(H, W)``.
        """
        image_np, class_mask_np = self.load_sample(index)

        image = tv_tensors.Image(
            torch.from_numpy(image_np)
            .permute(2, 0, 1)
            .contiguous()
        )
        mask = tv_tensors.Mask(
            torch.from_numpy(class_mask_np).contiguous()
        )

        # Drop numpy references before transform outputs are created.
        del image_np, class_mask_np

        if self.transform is not None:
            image, mask = self.transform(image, mask)
        else:
            image = image.to(dtype=torch.float32).div_(255.0)
            mask = mask.to(dtype=torch.int64)

        # Strip TVTensor subclasses at the dataset boundary. Downstream training
        # only needs ordinary tensors.
        image_tensor = image.as_subclass(torch.Tensor).contiguous()
        mask_tensor = mask.as_subclass(torch.Tensor).to(torch.int64).contiguous()

        return image_tensor, mask_tensor


## Utility Functions

In [ ]:
def find_image_path_from_mask(complete_mask_path: str, base_image_path: str) -> str:
    file_path_split = complete_mask_path.split("/")
    mask_file_name = file_path_split[-1].split("_")[0]

    image_path = base_image_path + "/" + mask_file_name + ".jpg"
    return image_path


def find_train_image_path_from_mask(complete_mask_path: str) -> str:
    return find_image_path_from_mask(complete_mask_path, Path.IMAGE_TRAIN_PATH)


def find_val_image_path_from_mask(complete_mask_path: str) -> str:
    return find_image_path_from_mask(complete_mask_path, Path.IMAGE_VAL_PATH)


def find_mask_path_from_image(complete_image_path: str, base_mask_path: str) -> str:
    file_path_split = complete_image_path.split("/")
    mask_file_name = file_path_split[-1].split(".")[0]

    mask_path = base_mask_path + "/" + mask_file_name + "_train_color.png"
    return mask_path


def find_train_mask_path_from_image(complete_image_path: str) -> str:
    return find_mask_path_from_image(complete_image_path, Path.SEGMENTATION_MASK_TRAIN_PATH)


def find_val_mask_path_from_image(complete_image_path: str) -> str:
    return find_mask_path_from_image(complete_image_path, Path.SEGMENTATION_MASK_VAL_PATH)


def load_dataset_from_files() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # load train, then split into train-test
    train_mask_paths = glob.glob(f"{Path.SEGMENTATION_MASK_TRAIN_PATH}/*.png")
    problematic_masks = []
    for complete_mask_path in train_mask_paths:
        with Image.open(complete_mask_path) as img:
            width, height = img.size
            if width != 1280 or height != 720:
                problematic_masks.append(complete_mask_path)
                train_mask_paths.remove(complete_mask_path)
    print(f"Problematic masks: {problematic_masks}")

    train_image_paths = list(map(find_train_image_path_from_mask, train_mask_paths))
    problematic_images = []
    for complete_image_path in train_image_paths:
        with Image.open(complete_image_path) as img:
            width, height = img.size
            if width != 1280 or height != 720:
                problematic_images.append(complete_image_path)
                train_image_paths.remove(complete_image_path)
                train_mask_paths.remove(find_train_mask_path_from_image(complete_image_path))
    print(f"Problematic images: {problematic_images}")

    train_test_df = pd.DataFrame({
        "image_paths": train_image_paths,
        "mask_paths": train_mask_paths,
    })
    train_df, test_df = train_test_split(train_test_df, test_size=0.2, random_state=Configuration.SEED)

    # load val
    val_mask_paths = glob.glob(f"{Path.SEGMENTATION_MASK_VAL_PATH}/*.png")
    problematic_val_masks = []
    for complete_mask_path in val_mask_paths:
        with Image.open(complete_mask_path) as img:
            width, height = img.size
            if width != 1280 or height != 720:
                problematic_val_masks.append(complete_mask_path)
                val_mask_paths.remove(complete_mask_path)
    print(f"Problematic val masks: {problematic_val_masks}")

    val_image_paths = list(map(find_val_image_path_from_mask, val_mask_paths))
    problematic_val_images = []
    for complete_image_path in val_image_paths:
        with Image.open(complete_image_path) as img:
            width, height = img.size
            if width != 1280 or height != 720:
                problematic_val_images.append(complete_image_path)
                val_image_paths.remove(complete_image_path)
                val_mask_paths.remove(find_val_mask_path_from_image(complete_image_path))
    print(f"Problematic val images: {problematic_val_images}")

    val_df = pd.DataFrame({
        "image_paths": val_image_paths,
        "mask_paths": val_mask_paths,
    })

    return train_df, val_df, test_df


def forward(model: nn.Module, x: BatchImage) -> Logits:
    """Execute a forward pass through the model and cast the result to Logits.

    Steps
    -----
    1. Run input tensor batch through the neural network.
    2. Return logits for downstream loss computation and type verification.

    Parameters
    ----------
    model : nn.Module
        Segmentation network mapping ``(B, 3, H, W)`` to ``(B, C, H, W)``[cite: 1].
    x : BatchImage
        Batch of input images of shape ``(B, 3, H, W)``.

    Returns
    -------
    Logits
        Raw output logits tensor of shape ``(B, NUM_CLASSES, H, W)``.
    """
    return model(x)


def kl_divergence(probs: Logits, adv_probs: Logits, eps: float = 1e-8) -> Scalar:
    """Compute mean per-pixel Kullback-Leibler divergence between predictions.

    TRADES measures adversarial perturbation impact by calculating the divergence
    between clean predictions and perturbed predictions[cite: 1]. Computing this per-pixel
    and averaging over batch and spatial dimensions keeps the regularizer scale
    invariant to input resolution.

    Steps
    -----
    1. Add ``eps`` to both probability distributions to prevent ``log(0)``.
    2. Sum ``probs * (log(probs) - log(adv_probs))`` over the class channel.
    3. Compute spatial and batch mean to yield a scalar divergence value.

    Parameters
    ----------
    probs : Logits
        Softmax probability tensor of clean predictions ``(B, C, H, W)``.
    adv_probs : Logits
        Softmax probability tensor of adversarial predictions ``(B, C, H, W)``.
    eps : float, optional
        Numerical stability epsilon to prevent undefined log evaluations. Defaults to 1e-8.

    Returns
    -------
    Scalar
        Averaged scalar KL-divergence tensor.
    """
    # Prevent undefined log values when probabilities are confident zeros
    kl_per_pixel = (probs * (torch.log(probs + eps) - torch.log(adv_probs + eps))).sum(dim=1)
    return kl_per_pixel.mean()

## Loss Functions

In [ ]:
class CompoundLoss(nn.Module):
    """Compound loss combining Dice Loss and Focal Loss for semantic segmentation.

    Semantic segmentation on class-imbalanced datasets benefits from combining a
    region-based loss (Dice loss) and a distribution-based loss (Focal loss).
    Dice loss optimizes overall mask overlap to handle class imbalance, while
    Focal loss down-weights easy background pixels to focus gradient updates on
    hard boundaries and minority classes.

    Steps
    -----
    1. Initialize the underlying SMP DiceLoss and FocalLoss modules in multiclass mode.
    2. In the forward pass, inspect target tensor dimensionality; if 4D (one-hot encoded),
       collapse the channel dimension to 2D class indices via ``argmax(dim=1)``.
    3. Compute multi-class Dice loss from raw logits.
    4. Compute multi-class Focal loss from raw logits.
    5. Return the weighted sum: ``dice_weight * dice + focal_weight * focal``.

    Parameters
    ----------
    dice_weight : float, optional
        Weight factor applied to the Dice loss component. Defaults to 0.5.
    focal_weight : float, optional
        Weight factor applied to the Focal loss component. Defaults to 1.0.
    ignore_index : int, optional
        Class index to ignore during loss computation. Defaults to 19.
    """

    def __init__(
        self,
        dice_weight: float = 0.5,
        focal_weight: float = 1.0,
        ignore_index: int = 19,
    ) -> None:
        super().__init__()
        # Store loss weighting factors to adjust the relative contribution of each loss term
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight

        # SMP DiceLoss operates directly on raw logits when from_logits=True,
        # avoiding an explicit external softmax step
        self.dice_loss = smp.losses.DiceLoss(
            mode=smp.losses.MULTICLASS_MODE,
            from_logits=True,
            ignore_index=ignore_index,
        )

        # SMP FocalLoss handles multiclass cross-entropy while masking out
        # the designated unknown/ignored class id
        self.focal_loss = smp.losses.FocalLoss(
            mode=smp.losses.MULTICLASS_MODE,
            ignore_index=ignore_index,
        )

    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:
        """Compute the weighted compound loss between predictions and targets.

        Steps
        -----
        1. Check if targets are 4D (one-hot encoded); if so, collapse to class indices.
        2. Evaluate Dice loss and Focal loss independently.
        3. Weight and sum both loss terms.

        Parameters
        ----------
        logits : Tensor
            Raw unnormalized model predictions of shape ``(B, C, H, W)``.
        targets : Tensor
            Ground-truth masks, either one-hot encoded tensors of shape ``(B, C, H, W)``
            or class indices of shape ``(B, H, W)``.

        Returns
        -------
        Tensor
            Scalar compound loss tensor suitable for gradient backpropagation.
        """
        # Collapse one-hot targets (B, C, H, W) to class indices (B, H, W) because
        # SMP multiclass loss functions with ignore_index require integer class indices
        if targets.ndim == 4:
            targets = targets.argmax(dim=1)

        dice = cast(Scalar, self.dice_loss(logits, targets))
        focal = cast(Scalar, self.focal_loss(logits, targets))

        # Combine weighted losses to balance boundary refinement and region overlap
        return self.dice_weight * dice + self.focal_weight * focal



class TradesLoss(nn.Module):
    """Memory-stable TRADES adversarial loss for semantic segmentation.

    The mathematical objective is unchanged from the original implementation:
        natural_loss(clean) + beta * KL(clean || adversarial).

    During the PGD inner loop, only the gradient with respect to the input
    ``x_adv`` is required. Model-parameter gradients are therefore disabled
    temporarily while crafting the adversarial example. They are restored
    before the final robustness forward pass and before the caller invokes
    ``loss.backward()``. This avoids building/storing parameter-gradient state
    for the repeated PGD forward passes.

    Parameters
    ----------
    natural_loss : nn.Module
        Loss applied to clean logits.
    epsilon : float, optional
        Radius of the L-inf adversarial perturbation ball. Defaults to 0.03.
    alpha : float, optional
        PGD step size. Defaults to 0.01.
    num_steps : int, optional
        Number of PGD iterations. Defaults to 10.
    beta : float, optional
        Robustness-loss weight. Defaults to 6.0.
    """

    def __init__(
        self,
        natural_loss: nn.Module,
        epsilon: float = 0.03,
        alpha: float = 0.01,
        num_steps: int = 10,
        beta: float = 6.0,
    ) -> None:
        super().__init__()
        self.natural_loss = natural_loss
        self.epsilon = epsilon
        self.alpha = alpha
        self.num_steps = num_steps
        self.beta = beta

    def forward(
        self,
        model: nn.Module,
        x: BatchImage,
        y: BatchIndexMask,
    ) -> tuple[Scalar, Logits]:
        """Compute the TRADES objective and detached clean logits."""

        # Clean branch is kept differentiable because the final loss.backward()
        # must propagate the natural-loss gradient into model parameters.
        clean_logits = forward(model, x)
        clean_probs = torch.softmax(clean_logits, dim=1).detach()

        was_training = model.training
        parameters = list(model.parameters())
        requires_grad_state = [parameter.requires_grad for parameter in parameters]

        # PGD needs d(KL)/dx_adv, but does not need d(KL)/d(theta).  Freezing
        # parameters during the inner loop substantially reduces autograd state.
        model.eval()
        for parameter in parameters:
            parameter.requires_grad_(False)

        try:
            x_adv = x.detach().clone()

            for _ in range(self.num_steps):
                x_adv.requires_grad_(True)

                adv_logits = forward(model, x_adv)
                adv_probs = torch.softmax(adv_logits, dim=1)
                kl = kl_divergence(clean_probs, adv_probs)

                grad = torch.autograd.grad(
                    kl,
                    x_adv,
                    create_graph=False,
                    retain_graph=False,
                )[0].detach()

                # The perturbation update itself does not need an autograd graph.
                with torch.no_grad():
                    perturbation = x_adv + self.alpha * grad.sign() - x
                    perturbation.clamp_(-self.epsilon, self.epsilon)
                    x_adv = torch.clamp(
                        x + perturbation,
                        0.0,
                        1.0,
                    ).detach()

                # Drop references to the attack graph before the next iteration.
                del adv_logits, adv_probs, kl, grad, perturbation

        finally:
            # Restore parameter gradient flags and training/eval mode exactly as
            # they were before the PGD attack.
            for parameter, requires_grad in zip(parameters, requires_grad_state, strict=True):
                parameter.requires_grad_(requires_grad)

            if was_training:
                model.train()
            else:
                model.eval()

        # Final adversarial branch is differentiable. Parameter gradients are
        # enabled again so loss.backward() can update the model.
        adv_logits = forward(model, x_adv)
        adv_probs = torch.softmax(adv_logits, dim=1)
        robust = kl_divergence(clean_probs, adv_probs)

        total_loss = self.natural_loss(clean_logits, y) + self.beta * robust

        # No second clean forward is needed by the training loop for metrics.
        return total_loss, clean_logits.detach()



## Training and Evaluation

In [ ]:
def report_runtime_memory(tag: str = "") -> None:
    """Print process, worker, /dev/shm, and CUDA memory usage.

    This diagnostic is intentionally based on host RSS as well as CUDA
    allocator statistics because the Kaggle failure message is a process-level
    memory failure, not a Python ``RuntimeError: CUDA out of memory``.
    """
    try:
        process = psutil.Process(os.getpid())
        rss_gib = process.memory_info().rss / (1024 ** 3)

        child_rss_gib = 0.0
        for child in process.children(recursive=True):
            try:
                child_rss_gib += child.memory_info().rss / (1024 ** 3)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass

        vm = psutil.virtual_memory()
        available_gib = vm.available / (1024 ** 3)

        print(
            f"[memory] {tag} | "
            f"main RSS={rss_gib:.2f} GiB | "
            f"child RSS={child_rss_gib:.2f} GiB | "
            f"RAM available={available_gib:.2f} GiB",
            flush=True,
        )

        try:
            shm = shutil.disk_usage("/dev/shm")
            print(
                f"[memory] /dev/shm used={shm.used / (1024 ** 3):.2f} GiB / "
                f"{shm.total / (1024 ** 3):.2f} GiB",
                flush=True,
            )
        except OSError:
            pass

        if torch.cuda.is_available():
            for device_index in range(torch.cuda.device_count()):
                torch.cuda.synchronize(device_index)
                allocated = torch.cuda.memory_allocated(device_index) / (1024 ** 3)
                reserved = torch.cuda.memory_reserved(device_index) / (1024 ** 3)
                free, total = torch.cuda.mem_get_info(device_index)
                print(
                    f"[memory] cuda:{device_index} | "
                    f"allocated={allocated:.2f} GiB | "
                    f"reserved={reserved:.2f} GiB | "
                    f"free={free / (1024 ** 3):.2f} / "
                    f"{total / (1024 ** 3):.2f} GiB",
                    flush=True,
                )
    except Exception as exc:
        print(f"[memory] diagnostic failed: {exc!r}", flush=True)


@torch.no_grad()
def compute_batch_macro_iou(
    y_pred: torch.Tensor,      # (B, C, H, W) logits
    y_true: torch.Tensor,      # (B, C, H, W) one-hot
    num_classes: int = 19,     # Classes 0 to 18 (excludes 19: Unknown)
    eps: float = 1e-7,
) -> float:
    """Compute the mean macro IoU over a batch, excluding ignored classes.

    The model outputs raw logits while the ground truth is one-hot, so both are
    first reduced to a single class id per pixel via ``argmax``. IoU is then
    computed class by class and averaged only over the classes that actually
    occur in either the prediction or the ground truth; empty classes are
    skipped rather than counted as zero, which would otherwise drag the mean
    down on batches dominated by a few classes.

    Steps
    -----
    1. Extract class predictions by finding argmax over the logits channel axis.
    2. For each class ID in ``[0, num_classes)``, calculate intersection and union masks.
    3. Exclude classes that appear in neither ground-truth nor prediction to avoid skewing the average.
    4. Compute mean IoU across all present classes.

    Parameters
    ----------
    y_pred : Logits
        Predicted logits tensor of shape ``(B, C, H, W)``.
    y_true : BatchIndexMask
        Ground-truth class indices of shape ``(B, H, W)``.
    num_classes : int, optional
        Number of classes evaluated, excluding background/unknown. Defaults to 19.
    eps : float, optional
        Division smoothing constant to prevent zero-division errors. Defaults to 1e-7.

    Returns
    -------
    float
        Macro IoU score averaged across active classes in the batch.
    """
    preds = y_pred.argmax(dim=1)

    iou_per_class = []
    for cls in range(num_classes):
        pred_mask = preds == cls
        true_mask = y_true == cls

        intersection = (pred_mask & true_mask).sum().float().item()
        union = (pred_mask | true_mask).sum().float().item()

        # Only evaluate classes that actually occur in either prediction or ground truth
        if union > 0:
            iou_per_class.append((intersection + eps) / (union + eps))

    return float(np.mean(iou_per_class)) if iou_per_class else 0.0



def execute_epoch(
    model: nn.Module,
    dataloader: DataLoader[tuple[ImageTensor, IndexMaskTensor]],
    optimizer: torch.optim.Optimizer,
    loss_fn: TradesLoss,
    device: torch.device,
) -> tuple[float, float]:
    """Execute one training epoch with explicit tensor lifetime management."""

    model.train()
    train_loss = 0.0
    train_iou = 0.0

    memory_log_interval = max(1, getattr(Configuration, "MEMORY_LOG_INTERVAL", 100))

    for batch_index, (X, y) in enumerate(dataloader, start=1):
        X = X.to(device, non_blocking=(device.type == "cuda"))
        y = y.to(device, non_blocking=(device.type == "cuda"))

        # set_to_none=True avoids writing zero tensors into every parameter grad.
        optimizer.zero_grad(set_to_none=True)

        loss, clean_logits = loss_fn(model, X, y)
        loss.backward()
        optimizer.step()

        train_loss += float(loss.detach().item())
        train_iou += compute_batch_macro_iou(
            clean_logits,
            y,
            num_classes=19,
        )

        # Make object lifetimes explicit at the end of each iteration.
        del loss, clean_logits, X, y

        if batch_index % memory_log_interval == 0:
            report_runtime_memory(
                f"train batch {batch_index}/{len(dataloader)}"
            )

    return (
        train_loss / len(dataloader),
        train_iou / len(dataloader),
    )



def evaluate(
    model: nn.Module,
    dataloader: DataLoader[tuple[ImageTensor, IndexMaskTensor]],
    loss_fn: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """Evaluate clean performance metrics on validation data.

    Steps
    -----
    1. Set model to eval mode and enter ``torch.inference_mode`` context.
    2. Pass clean validation batches through the model without gradient tracking.
    3. Accumulate validation loss and macro IoU scores.
    4. Return normalized validation metrics.

    Parameters
    ----------
    model : nn.Module
        Segmentation model under evaluation.
    dataloader : DataLoader[tuple[ImageTensor, IndexMaskTensor]]
        Validation data loader.
    loss_fn : nn.Module
        Standard clean loss criterion.
    device : torch.device
        Hardware accelerator running evaluation.

    Returns
    -------
    tuple[float, float]
        Tuple of (mean_eval_loss, mean_eval_macro_iou).
    """
    model.eval()
    eval_loss, eval_iou = 0.0, 0.0

    # inference_mode provides greater memory savings and speedups over no_grad
    with torch.inference_mode():
        for X, y in dataloader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            y_pred = forward(model, X)
            loss = loss_fn(y_pred, y)

            eval_loss += loss.item()
            eval_iou += compute_batch_macro_iou(y_pred, y, num_classes=19)

    eval_loss = eval_loss / len(dataloader)
    eval_iou = eval_iou / len(dataloader)

    return eval_loss, eval_iou



def train(
    model: nn.Module,
    train_dataloader: DataLoader[tuple[ImageTensor, IndexMaskTensor]],
    eval_dataloader: DataLoader[tuple[ImageTensor, IndexMaskTensor]],
    optimizer: torch.optim.Optimizer,
    loss_fn: TradesLoss,
    eval_loss_fn: nn.Module,
    epochs: int,
    train_device: torch.device,
    eval_device: torch.device,
    scheduler: lr_scheduler.ReduceLROnPlateau | None = None,
) -> tuple[nn.Module, Dict[str, List[float]]]:
    """Train, validate, and restore the best model without retaining a CPU copy."""

    session: Dict[str, List[float]] = {
        "loss": [],
        "macro_iou_score": [],
        "eval_loss": [],
        "eval_macro_iou_score": [],
    }

    best_eval_loss = float("inf")
    best_checkpoint_path = "./model/unet_best_state.pt"
    os.makedirs(os.path.dirname(best_checkpoint_path), exist_ok=True)

    report_runtime_memory("before training")

    for epoch in tqdm(range(epochs)):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        train_loss, train_iou = execute_epoch(
            model=model,
            dataloader=train_dataloader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=train_device,
        )

        eval_loss, eval_iou = evaluate(
            model=model,
            dataloader=eval_dataloader,
            loss_fn=eval_loss_fn,
            device=eval_device,
        )

        raw_model = model.module if hasattr(model, "module") else model

        if eval_loss < best_eval_loss:
            best_eval_loss = eval_loss
            # Store the best weights on disk rather than keeping an additional
            # full CPU state_dict alive for the remainder of the notebook.
            torch.save(
                raw_model.state_dict(),
                best_checkpoint_path,
            )

        current_lr = optimizer.param_groups[0]["lr"]
        if scheduler is not None:
            scheduler.step(eval_loss)
            current_lr = optimizer.param_groups[0]["lr"]

        log_text = (
            f"loss: {train_loss:.4f} - train_macro_iou: {train_iou:.4f} - "
            f"eval_loss: {eval_loss:.4f} - eval_macro_iou_score: {eval_iou:.4f}"
        )
        print(
            log_text + (f" - lr: {current_lr}" if scheduler is not None else "")
        )

        session["loss"].append(train_loss)
        session["macro_iou_score"].append(train_iou)
        session["eval_loss"].append(eval_loss)
        session["eval_macro_iou_score"].append(eval_iou)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        report_runtime_memory(f"end epoch {epoch + 1}")

    if os.path.exists(best_checkpoint_path):
        raw_model = model.module if hasattr(model, "module") else model

        # Load directly to the training device so a second long-lived CPU copy
        # of the full model is not required at the end of training.
        best_state = torch.load(
            best_checkpoint_path,
            map_location=train_device,
        )
        raw_model.load_state_dict(best_state)
        del best_state

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    report_runtime_memory("after restoring best checkpoint")

    return raw_model, session



def plot_training_curves(
    history: Dict[str, List[float]],
    fig_size: tuple[int, int] = (20, 10)
) -> None:

    loss = np.array(history['loss'])
    val_loss = np.array(history['eval_loss'])

    iou = np.array(history['macro_iou_score'])
    val_iou = np.array(history['eval_macro_iou_score'])

    epochs = range(len(history['loss']))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=fig_size)

    # Plot loss
    ax1.plot(epochs, loss, label='training_loss', marker='o', color='C5')
    ax1.plot(epochs, val_loss, label='eval_loss', marker='o', color='C6')

    # Fill area between losses
    ax1.fill_between(epochs, loss, val_loss, where=(loss > val_loss), color='C5', alpha=0.4, interpolate=True)
    ax1.fill_between(epochs, loss, val_loss, where=(loss < val_loss), color='C6', alpha=0.4, interpolate=True)

    # Add Text & Formats
    ax1.set_title('Loss (Lower Means Better)', fontsize=22)
    ax1.set_xlabel('Epochs', fontsize=18)
    ax1.set_ylabel('Loss', fontsize=18)
    ax1.tick_params(axis='both', which='major', labelsize=14)
    ax1.legend(fontsize=14)

    # Plot metric
    ax2.plot(epochs, iou, label='training_macro_iou', marker='o', color='C5')
    ax2.plot(epochs, val_iou, label='eval_macro_iou', marker='o', color='C6')

    # Fill area between metrics
    ax2.fill_between(epochs, iou, val_iou, where=(iou > val_iou), color='C5', alpha=0.4, interpolate=True)
    ax2.fill_between(epochs, iou, val_iou, where=(iou < val_iou), color='C6', alpha=0.4, interpolate=True)

    # Add Text & Formats
    ax2.set_title('Macro IoU (Higher Means Better)', fontsize=22)
    ax2.set_xlabel('Epochs', fontsize=18)
    ax2.set_ylabel('Macro IoU', fontsize=18)
    ax2.tick_params(axis='both', which='major', labelsize=14)
    ax2.legend(fontsize=14)
    sns.despine()


def colorize_mask(class_mask: np.ndarray, palette: np.ndarray) -> np.ndarray:
    """Map a class-index mask to an RGB image using ``palette``.

    Steps
    -----
    1. Cast ``class_mask`` to integer so it can be used as row indices.
    2. Index ``palette`` with those indices, turning a ``(H, W)`` array of
       class ids into a ``(H, W, 3)`` image.

    Parameters
    ----------
    class_mask : np.ndarray
        Array of shape ``(H, W)`` whose values are class indices.
    palette : np.ndarray
        Array of shape ``(NUM_CLASSES, 3)`` mapping a class id to an RGB
        colour (the 0-255 range).

    Returns
    -------
    np.ndarray
        RGB image of shape ``(H, W, 3)`` matching the dtype of ``palette``.
    """
    return palette[class_mask.astype(np.int64)]


def visualize_predictions(
    model: nn.Module,
    test_df: pd.DataFrame,
    device: torch.device,
    num_samples: int = 4,
    output_path: str = "./predictions.png",
) -> None:
    """Render images, ground-truth masks, and model predictions.

    The same torchvision-v2 inference preprocessing used by validation is used
    here, avoiding a second preprocessing implementation.

    Parameters
    ----------
    model : nn.Module
        Trained segmentation model.
    test_df : pd.DataFrame
        DataFrame containing sample image and mask paths.
    device : torch.device
        Device used for inference.
    num_samples : int, optional
        Number of test samples to render. Defaults to 4.
    output_path : str, optional
        Destination PNG path. Defaults to ``./predictions.png``.

    Raises
    ------
    ValueError
        If ``test_df`` is empty.
    """
    sample_df = test_df.head(min(num_samples, len(test_df))).reset_index(drop=True)
    if sample_df.empty:
        raise ValueError("test_df has no rows to visualize.")

    inference_transform = v2.Compose(
        [
            v2.Resize(
                (Configuration.IMAGE_HEIGHT, Configuration.IMAGE_WIDTH),
                antialias=True,
            ),
            v2.ToDtype(
                {
                    tv_tensors.Image: torch.float32,
                    tv_tensors.Mask: torch.int64,
                    "others": None,
                },
                scale=True,
            ),
        ]
    )

    sample_ds = BDDSegmentationDataset(
        sample_df,
        transform=inference_transform,
    )

    eval_model = model.module if hasattr(model, "module") else model
    eval_model.eval()

    num_rows = len(sample_df)
    fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5 * num_rows))
    if num_rows == 1:
        axes = axes[np.newaxis, :]

    for row in range(num_rows):
        image_tensor, class_mask_tensor = sample_ds[row]

        with torch.inference_mode():
            logits = eval_model(image_tensor.unsqueeze(0).to(device))

        pred_class = logits.argmax(dim=1).squeeze(0).cpu().numpy()
        class_mask = class_mask_tensor.cpu().numpy()

        image = (
            image_tensor.permute(1, 2, 0)
            .cpu()
            .numpy()
            .clip(0.0, 1.0)
        )

        pred_color = (
            colorize_mask(pred_class, CLASS_COLORS).astype(np.float32) / 255.0
        )
        true_mask_color = (
            colorize_mask(class_mask, CLASS_COLORS).astype(np.float32) / 255.0
        )

        axes[row, 0].imshow(image)
        axes[row, 0].set_title("Image")

        axes[row, 1].imshow(image)
        axes[row, 1].imshow(true_mask_color, alpha=0.5)
        axes[row, 1].set_title("Image + True Mask")

        axes[row, 2].imshow(image)
        axes[row, 2].imshow(pred_color, alpha=0.5)
        axes[row, 2].set_title("Image + Predicted Mask")

        for ax in axes[row]:
            ax.set_xticks([])
            ax.set_yticks([])

        del logits, image_tensor, class_mask_tensor

    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


## Main

In [ ]:
# Print current Torch package versions
print("Package versions:")
print("*" * 26)
print(f"torch          - {torch.__version__}")
print(f"torchvision    - {torchvision.__version__}")

train_df, val_df, test_df = load_dataset_from_files()

# IMPORTANT:
# Albumentations 2.0.8 is intentionally not used here. The training hot path
# uses torchvision v2 TVTensors so geometric transforms are shared between the
# image and segmentation mask without the long-running host-RSS growth observed
# with the previous Albumentations pipeline.
train_transforms = v2.Compose(
    [
        v2.Resize(
            (Configuration.IMAGE_HEIGHT, Configuration.IMAGE_WIDTH),
            antialias=True,
        ),
        # Semantically comparable to A.RandomBrightnessContrast(p=0.2):
        # when selected, jitter brightness and contrast by up to +/-20%.
        v2.RandomApply(
            [v2.ColorJitter(brightness=0.2, contrast=0.2)],
            p=0.2,
        ),
        v2.RandomHorizontalFlip(p=0.5),
        v2.ToDtype(
            {
                tv_tensors.Image: torch.float32,
                tv_tensors.Mask: torch.int64,
                "others": None,
            },
            scale=True,
        ),
    ]
)

inference_transforms = v2.Compose(
    [
        v2.Resize(
            (Configuration.IMAGE_HEIGHT, Configuration.IMAGE_WIDTH),
            antialias=True,
        ),
        v2.ToDtype(
            {
                tv_tensors.Image: torch.float32,
                tv_tensors.Mask: torch.int64,
                "others": None,
            },
            scale=True,
        ),
    ]
)

train_ds = BDDSegmentationDataset(
    train_df,
    transform=train_transforms,
)
val_ds = BDDSegmentationDataset(
    val_df,
    transform=inference_transforms,
)

print(f"Reusing precomputed class weights from '{Path.CLASS_WEIGHTS_PATH}'...")
train_sample_weights = np.load(Path.CLASS_WEIGHTS_PATH)

train_sampler = WeightedRandomSampler(
    weights=train_sample_weights.tolist(),
    num_samples=len(train_ds),
    replacement=True,
)

# Keep loading in-process for the stability baseline. With Albumentations removed,
# NUM_WORKERS can later be raised experimentally, but it is not needed for correctness.
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=Configuration.BATCH_SIZE,
    sampler=train_sampler,
    num_workers=Configuration.NUM_WORKERS,
    pin_memory=False,
)
val_loader = DataLoader(
    dataset=val_ds,
    batch_size=Configuration.BATCH_SIZE,
    shuffle=False,
    num_workers=Configuration.NUM_WORKERS,
    pin_memory=False,
)

model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=Configuration.CHANNELS,
    classes=Configuration.NUM_CLASSES,
)
model = model.to(Configuration.DEVICE)

print(
    summary(
        model=model,
        input_size=(
            Configuration.BATCH_SIZE,
            Configuration.CHANNELS,
            Configuration.IMAGE_HEIGHT,
            Configuration.IMAGE_WIDTH,
        ),
        col_names=["output_size", "num_params", "trainable"],
        col_width=30,
        row_settings=["var_names"],
        depth=5,
    )
)

if torch.cuda.device_count() > 1:
    print(
        f"{torch.cuda.device_count()} GPUs detected. "
        "nn.DataParallel is intentionally DISABLED for the stability baseline; "
        "training uses cuda:0 only."
    )

natural_loss = CompoundLoss(
    dice_weight=0.5,
    focal_weight=1.0,
    ignore_index=19,
)

loss_fn = TradesLoss(natural_loss=natural_loss)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=Configuration.LR,
)

scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer,
    mode="min",
    patience=Configuration.PATIENCE,
)

print("Training U-Net Model")
print(f"Train on {len(train_df)} samples, validate on {len(val_df)} samples.")
print("----------------------------------")

session_config = {
    "model": model,
    "train_dataloader": train_loader,
    "eval_dataloader": val_loader,
    "optimizer": optimizer,
    "scheduler": scheduler,
    "loss_fn": loss_fn,
    "eval_loss_fn": loss_fn.natural_loss,
    "epochs": Configuration.EPOCHS,
    "train_device": Configuration.DEVICE,
    "eval_device": Configuration.DEVICE,
}

model, unet_session_history = train(**session_config)

model_name = "teacher"
model_path = "./model/"
os.makedirs(model_path, exist_ok=True)

# Save a portable state_dict rather than pickling the entire model object.
torch.save(
    model.state_dict(),
    os.path.join(model_path, f"{model_name}_state_dict.pt"),
)

unet_session_history_df = pd.DataFrame(unet_session_history)
print(unet_session_history_df)

plot_training_curves(
    unet_session_history,
    fig_size=(20, 20),
)

visualize_predictions(
    model,
    test_df,
    Configuration.DEVICE,
)
